\setcounter{secnumdepth}{0}

# Mutatie bestand #

[naam bestand plus info wat het bevat]

#### Stap 1. Inlezen en verkennen van de mutatie-data


Doel: de mutatie-data van CCLE inlezen en een eerste verkenning doen.

In [51]:
import pandas as pd

# aanmaken variabele met de bestandpad
mutatie_file = "../data/raw/OmicsSomaticMutations.csv"

# inlezen 
mutatie_df = pd.read_csv(mutatie_file, low_memory=False) # csv, geen tabs

# verkenning
mutatie_df.info()
print(mutatie_df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 751310 entries, 0 to 751309
Data columns (total 70 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Chrom                            751310 non-null  object 
 1   Pos                              751310 non-null  int64  
 2   Ref                              751310 non-null  object 
 3   Alt                              751310 non-null  object 
 4   AF                               751310 non-null  float64
 5   DP                               751310 non-null  int64  
 6   RefCount                         751310 non-null  int64  
 7   AltCount                         751310 non-null  int64  
 8   GT                               751310 non-null  object 
 9   PS                               44923 non-null   float64
 10  VariantType                      751310 non-null  object 
 11  VariantInfo                      751310 non-null  object 
 12  DN

De mutatie-dataset bevat 751310 rijen en 70 kolommen en bestaat uit de volgende datatypen:

- object: tekst/categorieën (bijvoorbeeld genen en mutatietypes)
- float64: decimale waarden
- bool: waar/onwaar waarden (True/False)
- int64: hele getallen (bijvoorbeeld aantallen mutaties)

**Belangrijk voor dit onderzoek:**  
ModelID: koppeling tussen datasets.  
HugoSymbol/EnsemblGeneID: gen waarin mutatie voorkomt.  
VariantType: type mutatie (feature voor ML).  
DNAChange/ProteinChange: optioneel voor interpretatie.   
AF, RefCount, AltCount: betrouwbaarheid / filteren.   
LikelyLoF, OncogeneHighImpact, TumorSuppressorHighImpact: impact op functie/belangrijke features.    
Sift, Polyphen, RevelScore: schadelijkheidsvoorspelling.  
PrimaryDisease/Tissue: filteren op longkankercellijnen.



#### Stap 2. Filteren op relevante data


Doel: alleen de biologische en analyse-relevante mutaties behouden, zodat de dataset overzichtelijk blijft en rekentijd beperkt blijft.

Acties:
1. Filteren op cellijntype: alleen cellijnen van het weefseltype "lung" worden behouden.
2. Filteren op relevante mutaties: niet alle mutaties zijn biologisch significant (veel varianten zijn neutraal of technisch artefact). Daarom worden alleen functionele mutaties behouden op basis van de volgende kolommen:
   - LiklyLoF (verlies-van-functie mutaties)
   - OncogeneHighImpact (activerende mutaties in oncogenen)
   - TumorSuppressorHighImpact (verlies-van-remfunctie in tumorsuppressorgen)
   - Hotspot (bekende activerende hotspot-mutaties)
   - HessDriver (mutaties geidentificeerd als drivers)

Een mutatie wordt behouden als één of meer van deze criteria waar zijn.


In [52]:
# kolommen waar long kan voorkomen
tissue_cols = ['Site_Primary', 'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3']

# inlezen van het metadata-bestand
metadata_file = "../data/raw/Cell_lines_annotations_20181226.txt"
metadata_df = pd.read_csv(metadata_file, sep='\t', low_memory=False) # txt, tab seperated

# voor check: aantal rijen en unieke cellijnen in originele mutatie_df
print(f"Aantal rijen in originele mutatie_df: {mutatie_df.shape[0]}")
print(f"Aantal unieke cellijnen in originele mutatie_df: {mutatie_df['ModelID'].nunique()}")

# filteren op long 
lung_meta = metadata_df[metadata_df[tissue_cols].apply(lambda x: x.str.contains('lung', case = False, na = False)).any(axis=1)
]

# depMapIDs van longkankercellijnen ophalen
lung_depmap_ids = lung_meta['depMapID'].tolist()

# filter mutatie_df op longkankercellijnen
lung_mutatie_df = mutatie_df[mutatie_df['ModelID'].isin(lung_depmap_ids)]

# filter alleen biologisch relevante mutaties
filtered = lung_mutatie_df[
    lung_mutatie_df['LikelyLoF'] |
    lung_mutatie_df['OncogeneHighImpact'] |
    lung_mutatie_df['TumorSuppressorHighImpact'] |
    lung_mutatie_df['Hotspot'] |
    lung_mutatie_df['HessDriver']
]

# na check: aantal rijen en unieke cellijnen na filtering
print(f"Aantal rijen na filtering: {filtered.shape[0]}")
print(f"Aantal unieke cellijnen: {filtered['ModelID'].nunique()}")

Aantal rijen in originele mutatie_df: 751310
Aantal unieke cellijnen in originele mutatie_df: 1939
Aantal rijen na filtering: 12630
Aantal unieke cellijnen: 174


#### Stap 3. Quality control / opschonen

**Doel:** Zorg dat de dataset geschikt is voor machine learning

**Acties:**

1. Verwijder genen die in alle cellijnen geen mutaties hebben (alleen 0).  
2. Controleer dat alle cellijnen en genen uniek zijn, zonder duplicaten.  
3. Controleer op ontbrekende waarden:  
   - NaN betekent meestal geen mutatie --> invullen met 0.

In [53]:
#check op echte ontbrekende waarden in originele mutatie_df
missing_modelid = mutatie_df['ModelID'].isna().sum()
missing_gene = mutatie_df['HugoSymbol'].isna().sum()
print(f"Echte ontbrekende waarden vóór filtering:")
print(f" - Ontbrekende ModelID: {missing_modelid}")
print(f" - Ontbrekende genen (HugoSymbol): {missing_gene}\n")

# basisoverzicht van filtered dataframe
num_cell_lines = filtered['ModelID'].nunique()
num_genes = filtered['HugoSymbol'].nunique()
num_mutations = filtered.shape[0]
total_combinations = num_cell_lines * num_genes
percent_mutated = 100 * num_mutations / total_combinations

print(f"Aantal cellijnen: {num_cell_lines}")
print(f"Aantal genen: {num_genes}")
print(f"Aantal mutaties in dataset: {num_mutations}")
print(f"Totaal mogelijke combinaties (cellijnen x genen): {total_combinations}")
print(f"Percentage mutaties aanwezig: {percent_mutated:.2f}%")
print("De overige combinaties zijn geen mutaties, wat biologisch correct is.\n")

# 2. pivot naar matrix zonder fill_value, zodat echte NaN's zichtbaar zijn
temp_matrix = (
    filtered
    .assign(mutated=1)
    .pivot_table(index='ModelID', columns='HugoSymbol', values='mutated')
)

# controleer aantal NaN's in de pivot
missing_count_before = temp_matrix.isna().sum().sum()
print(f"Aantal ontbrekende waarden in pivot vóór invullen: {missing_count_before}")
print("Deze NaN's vertegenwoordigen cellijn-gen combinaties zonder geregistreerde mutaties.\n")

# invullen van NaN's met 0 voor definitieve binaire matrix
mut_matrix_qc = temp_matrix.fillna(0)

# controleer opnieuw
missing_count_after = mut_matrix_qc.isna().sum().sum()
print(f"Aantal ontbrekende waarden na invullen: {missing_count_after}")


Echte ontbrekende waarden vóór filtering:
 - Ontbrekende ModelID: 0
 - Ontbrekende genen (HugoSymbol): 0

Aantal cellijnen: 174
Aantal genen: 7358
Aantal mutaties in dataset: 12630
Totaal mogelijke combinaties (cellijnen x genen): 1280292
Percentage mutaties aanwezig: 0.99%
De overige combinaties zijn geen mutaties, wat biologisch correct is.

Aantal ontbrekende waarden in pivot vóór invullen: 1267889
Deze NaN's vertegenwoordigen cellijn-gen combinaties zonder geregistreerde mutaties.

Aantal ontbrekende waarden na invullen: 0


In de originele dataset zijn geen echte ontbrekende waarden aanwezig in de sleutelkolommen (ModelID en HugoSymbol). In de pivotmatrix ontstaan veel NaN's omdat alleen combinaties met geregistreeerde mutaties een waarde hebben. Deze NaN's zijn ingevuld met 0 voor de definitieve binaire matrix, wat aangeeft dat het gen in die cellijn geen relevante mutatie bevat. Het lage percentage mutaties (~0.99%) is biologisch consistent met wat wordt verwacht in longkankercellijnen en draagt daarmee bij aan de betrouwbaarheid.

#### Stap 4. Transformatie naar binaire matrix


**Doel:** 
De dataset geschikt maken voor machine learning:   
- rijen = cellijnen  
- kolommen = genen  
- waarden = 0/1 (binair)  

**Acties:**  
- Voor elke cellijn (ModelID) en gen (HugoSymbol):  
    - 1 als minstens één relevante mutatie aanwezig is  
    - 0 als geen mutatie aanwezig is

In [54]:
# binaire matrix maken
mut_matrix = mut_matrix_qc.astype(int)

# check shape en eerste paar kolommen
print(mut_matrix.shape)
print(mut_matrix.iloc[:, :5].head())


(174, 7358)
HugoSymbol  A1CF  A2M  A2ML1  A4GALT  AAAS
ModelID                                   
ACH-000012     0    0      0       0     0
ACH-000015     0    0      0       0     0
ACH-000021     0    0      0       0     0
ACH-000029     0    0      0       0     0
ACH-000030     0    0      0       0     0


In [55]:
import os

# output map voor verwerkte data (hier komt de pickle terecht)
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# sla de geharmoniseerde responsmatrix op als pickle
mut_matrix.to_pickle(
    os.path.join(output_dir, "mut_matrix.pkl"))
